# Adaptive RAG Application with FAISS, CrossEncoder Reranking and an OpenAI-Compatible Client

This notebook implements a transparent Retrieval-Augmented Generation (RAG) system for question answering over an uploaded PDF.

The code is deliberately human-readable, while the explanatory discussion is written at a **Master's-level academic standard**.

## Important API clarification

This project imports the official-style client with:

```python
from openai import OpenAI
```

However, an API key only authenticates against the provider that issued it.

Because this notebook is configured to use your:

```text
DEEPSEEK_API_KEY
```

the `OpenAI` Python client is used as an **OpenAI-compatible interface**, while the request is sent to the DeepSeek API endpoint.

A DeepSeek API key cannot directly call OpenAI-hosted GPT models. If you later want to use a genuine OpenAI GPT model, you must supply an `OPENAI_API_KEY` and change the provider configuration cell.

## Architecture

```text
PDF upload
    ↓
Document profiling
    ↓
Automatic chunk-strategy selection
    ↓
Sentence-aware adaptive chunking
    ↓
Embedding generation
    ↓
FAISS candidate retrieval
    ↓
CrossEncoder reranking
    ↓
Best evidence selection
    ↓
Prompt augmentation
    ↓
OpenAI-compatible LLM client
    ↓
Grounded answer + sources
```

Unlike a fixed-chunk demonstration, this notebook first **profiles the uploaded PDF** and selects its chunking parameters from the observed document characteristics.

## Step 1 — Install the required libraries

The notebook uses PyMuPDF for PDF extraction, Sentence Transformers for dense embeddings and reranking, FAISS for vector retrieval, and the OpenAI Python SDK as the client interface.

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu openai python-dotenv

print("✅ Required packages installed")

## Step 2 — Upload your `.env` file

Your `.env` file should contain:

```text
DEEPSEEK_API_KEY=your_actual_deepseek_api_key
```

The notebook expects it at:

```text
/content/.env
```

In [ ]:
from google.colab import files
import os

print("Upload your .env file:")
uploaded_env = files.upload()

print("\nUploaded:")
for file_name in uploaded_env:
    print(" -", file_name)

print("\nCurrent /content files:")
print(os.listdir("/content"))

## Step 3 — Upload the PDF that will form the RAG knowledge base

The PDF filename does not need to be hard-coded. The notebook detects the uploaded PDF automatically.

In [ ]:
from google.colab import files
import os

print("Upload one PDF:")
uploaded_pdf = files.upload()

pdf_files = [
    file_name
    for file_name in uploaded_pdf
    if file_name.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("❌ No PDF file was detected.")

pdf_name = pdf_files[0]
pdf_path = f"/content/{pdf_name}"

print("\n✅ PDF detected")
print("Filename :", pdf_name)
print("Path     :", pdf_path)
print("Exists?  :", os.path.exists(pdf_path))

## Step 4 — Import the libraries

In [ ]:
import os
import re
import math
import statistics

import fitz
import numpy as np
import faiss

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

print("✅ Libraries imported")

## Step 5 — Load the API key and configure the LLM client

The variable is deliberately called `llm_client`, rather than `deepseek`, because the application code uses the OpenAI-compatible client abstraction.

### Current configuration

```text
Credential: DEEPSEEK_API_KEY
Client:     OpenAI Python SDK
Endpoint:   https://api.deepseek.com
```

The model identifier still has to be one supported by the DeepSeek endpoint because the DeepSeek API key does not provide access to OpenAI-hosted GPT models.

In [ ]:
env_path = "/content/.env"

if not os.path.exists(env_path):
    raise FileNotFoundError(
        "❌ /content/.env was not found. "
        "Upload a file named exactly .env"
    )

load_dotenv(env_path)

api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError(
        "❌ DEEPSEEK_API_KEY was not found inside .env"
    )

llm_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

GENERATION_MODEL = "deepseek-v4-flash"

print("✅ OpenAI-compatible client configured")
print("API key loaded safely")
print("Generation endpoint: https://api.deepseek.com")

### Optional future switch to a genuine OpenAI GPT model

Do **not** run the following configuration unless you have an OpenAI API key.

For example:

```python
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

llm_client = OpenAI(api_key=OPENAI_API_KEY)
GENERATION_MODEL = "gpt-5.6-luna"
```

That is a provider change, not merely a model-name change.

## Step 6 — Test the API connection

In [ ]:
print("Testing the language-model connection...")

test_response = llm_client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Connection successful"
        }
    ]
)

print("\nModel response:")
print(test_response.choices[0].message.content)

# Part A — Scan and profile the PDF

Before selecting chunk parameters, the notebook first examines the structure and size of the uploaded document.

## Step 7 — Extract text page by page

In [ ]:
pdf = fitz.open(pdf_path)

pages = []

for page_number, page in enumerate(pdf, start=1):
    text = page.get_text("text").strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

pdf.close()

print("✅ PDF scan complete")
print("Total PDF pages       :", len(pdf) if False else "read successfully")
print("Pages containing text :", len(pages))

if pages:
    print("\nFirst-page preview:")
    print(pages[0]["text"][:600])

### MSc-level rationale

PDF ingestion is not simply a file-reading operation. PDF is fundamentally a layout-oriented format, so extraction quality can vary depending on whether the source contains native text, scanned images, multi-column layouts, tables, headers, or embedded figures.

The present implementation uses native text extraction because the objective is to study the RAG pipeline itself. A production-grade ingestion layer would normally add OCR fallback, layout detection, table extraction, metadata parsing, and extraction-quality checks.

## Step 8 — Clean the extracted text

In [ ]:
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


for page in pages:
    page["text"] = clean_text(page["text"])

print("✅ Text cleaned")

if pages:
    print("\nCleaned preview:")
    print(pages[0]["text"][:600])

### MSc-level rationale

Cleaning is intentionally conservative. Semantic embedding models benefit from natural linguistic structure, so aggressive stemming or stop-word removal is usually unnecessary.

The objective is to remove extraction noise while preserving domain-specific terminology, numbers, section labels, dates, and sentence structure that may later contribute to retrieval relevance.

## Step 9 — Profile the document automatically

The notebook now measures several characteristics of the uploaded PDF:

- number of text-bearing pages
- total characters
- total words
- approximate token count
- average characters per page
- paragraph count
- average paragraph length
- median paragraph length

These measurements are then used to select a chunking strategy.

In [ ]:
all_text = "\n\n".join(page["text"] for page in pages)

paragraphs = [
    paragraph.strip()
    for paragraph in re.split(r"\n\s*\n", all_text)
    if paragraph.strip()
]

paragraph_lengths = [len(p) for p in paragraphs]

page_count = len(pages)
total_characters = len(all_text)
total_words = len(all_text.split())

# A rough approximation for English prose.
estimated_tokens = max(1, round(total_characters / 4))

average_chars_per_page = (
    round(total_characters / page_count)
    if page_count
    else 0
)

average_paragraph_length = (
    round(statistics.mean(paragraph_lengths))
    if paragraph_lengths
    else 0
)

median_paragraph_length = (
    round(statistics.median(paragraph_lengths))
    if paragraph_lengths
    else 0
)

document_profile = {
    "page_count": page_count,
    "total_characters": total_characters,
    "total_words": total_words,
    "estimated_tokens": estimated_tokens,
    "average_chars_per_page": average_chars_per_page,
    "paragraph_count": len(paragraphs),
    "average_paragraph_length": average_paragraph_length,
    "median_paragraph_length": median_paragraph_length
}

print("=" * 72)
print("AUTOMATIC DOCUMENT PROFILE")
print("=" * 72)

for key, value in document_profile.items():
    print(f"{key:28} : {value}")

### MSc-level rationale

A universal chunk size is rarely optimal because document characteristics vary substantially.

For example:

- a 5-page policy document may benefit from relatively compact chunks,
- a 150-page technical manual may require larger chunks to avoid creating an unnecessarily large vector collection,
- documents dominated by long paragraphs may need larger chunk windows to preserve local semantic coherence,
- sparse presentation-style PDFs may require smaller retrieval units.

The method used here is **adaptive but heuristic**. It does not claim to discover a mathematically optimal chunk size. Instead, it makes a reproducible decision from observable document statistics.

This distinction is important at Master's level: automatic parameter selection should be understood as a modelled engineering heuristic unless it has been optimised against a labelled evaluation set.

# Part B — Automatically select chunking parameters

## Step 10 — Let the notebook choose the chunk strategy

The function below uses the document profile to determine:

- target chunk size
- overlap
- a short textual explanation of the decision

The decision is made after the PDF has been scanned rather than being hard-coded in advance.

In [ ]:
def choose_chunk_strategy(profile):
    pages = profile["page_count"]
    chars = profile["total_characters"]
    median_para = profile["median_paragraph_length"]
    avg_page = profile["average_chars_per_page"]

    # ---------------------------------------------------------
    # 1. Choose a base size from overall document scale.
    # ---------------------------------------------------------
    if pages <= 10 and chars <= 30_000:
        target_size = 700
        scale_reason = "small document"

    elif pages <= 40 and chars <= 120_000:
        target_size = 1000
        scale_reason = "medium-sized document"

    elif pages <= 120 and chars <= 400_000:
        target_size = 1400
        scale_reason = "large document"

    else:
        target_size = 1800
        scale_reason = "very large document"

    # ---------------------------------------------------------
    # 2. Adjust for paragraph structure.
    # Long paragraphs suggest that slightly larger chunks may
    # preserve local semantic coherence better.
    # ---------------------------------------------------------
    if median_para > 700:
        target_size += 250
        paragraph_reason = "long paragraph structure"

    elif 0 < median_para < 180:
        target_size -= 150
        paragraph_reason = "short paragraph structure"

    else:
        paragraph_reason = "moderate paragraph structure"

    # ---------------------------------------------------------
    # 3. Adjust for page density.
    # ---------------------------------------------------------
    if avg_page > 5000:
        target_size += 150
        density_reason = "high page density"

    elif 0 < avg_page < 1200:
        target_size -= 100
        density_reason = "low page density"

    else:
        density_reason = "moderate page density"

    # Keep the final choice within sensible educational bounds.
    target_size = max(600, min(target_size, 2200))

    # Use approximately 15% overlap.
    overlap = round(target_size * 0.15)

    return {
        "target_chunk_chars": target_size,
        "overlap_chars": overlap,
        "reason": (
            f"{scale_reason}; "
            f"{paragraph_reason}; "
            f"{density_reason}"
        )
    }


chunk_strategy = choose_chunk_strategy(document_profile)

print("=" * 72)
print("AUTOMATIC CHUNKING DECISION")
print("=" * 72)
print("Target chunk size :", chunk_strategy["target_chunk_chars"], "characters")
print("Overlap           :", chunk_strategy["overlap_chars"], "characters")
print("Reason            :", chunk_strategy["reason"])

### MSc-level rationale

The adaptive strategy uses three classes of evidence:

1. **Corpus scale** — controls the balance between retrieval granularity and index size.
2. **Paragraph structure** — acts as a proxy for local semantic-unit length.
3. **Page density** — provides additional information about how much textual content is concentrated on each page.

The chosen overlap is approximately 15% of the target chunk size. Overlap introduces redundancy, but it reduces the probability that evidence spanning a boundary will be split into completely disconnected retrieval units.

The trade-off can be expressed conceptually as:

\[
\text{larger chunks}
\rightarrow
\text{more context per retrieval unit but lower topical specificity}
\]

whereas:

\[
\text{smaller chunks}
\rightarrow
\text{higher specificity but greater risk of semantic fragmentation}
\]

## Step 11 — Perform sentence-aware adaptive chunking

Instead of blindly cutting text every fixed number of characters, this chunker first identifies sentence boundaries and then packs sentences until the automatically selected target size is reached.

This reduces mid-sentence fragmentation.

In [ ]:
def split_into_sentences(text):
    # Simple sentence-boundary heuristic suitable for an educational project.
    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


sentence_units = []

for page in pages:
    for sentence in split_into_sentences(page["text"]):
        sentence_units.append({
            "text": sentence,
            "page": page["page"]
        })

print("Sentence units detected:", len(sentence_units))

In [ ]:
def make_adaptive_chunks(
    sentence_units,
    target_chars,
    overlap_chars
):
    chunks = []
    current_units = []
    current_length = 0

    def save_chunk(units):
        text = " ".join(unit["text"] for unit in units).strip()
        pages_used = sorted(set(unit["page"] for unit in units))

        if text:
            chunks.append({
                "chunk_id": len(chunks),
                "pages": pages_used,
                "text": text
            })

    for unit in sentence_units:
        sentence = unit["text"]
        sentence_length = len(sentence)

        # If adding the next sentence would make the current chunk
        # substantially larger than the target, save the current chunk.
        if (
            current_units
            and current_length + sentence_length > target_chars
        ):
            save_chunk(current_units)

            # Create overlap by keeping sentences from the end of
            # the previous chunk until roughly overlap_chars is reached.
            overlap_units = []
            overlap_length = 0

            for old_unit in reversed(current_units):
                overlap_units.insert(0, old_unit)
                overlap_length += len(old_unit["text"])

                if overlap_length >= overlap_chars:
                    break

            current_units = overlap_units
            current_length = sum(
                len(item["text"])
                for item in current_units
            )

        current_units.append(unit)
        current_length += sentence_length

    if current_units:
        save_chunk(current_units)

    return chunks


chunks = make_adaptive_chunks(
    sentence_units=sentence_units,
    target_chars=chunk_strategy["target_chunk_chars"],
    overlap_chars=chunk_strategy["overlap_chars"]
)

print("✅ Adaptive chunking complete")
print("Total chunks created:", len(chunks))

if chunks:
    chunk_lengths = [len(chunk["text"]) for chunk in chunks]

    print("Average chunk length:", round(statistics.mean(chunk_lengths)))
    print("Median chunk length :", round(statistics.median(chunk_lengths)))
    print("Smallest chunk      :", min(chunk_lengths))
    print("Largest chunk       :", max(chunk_lengths))

### MSc-level rationale

Sentence-aware packing is a compromise between fixed-width chunking and more computationally expensive semantic segmentation.

It preserves sentence integrity while remaining deterministic and inexpensive. The resulting chunks may still combine multiple related ideas, but they are less likely to contain abrupt syntactic truncation.

A more advanced dissertation-level extension could compare this heuristic against:
- recursive chunking,
- embedding-based semantic break detection,
- topic segmentation,
- heading-aware hierarchical chunking,
- proposition-based chunking.

The correct choice should ultimately be determined empirically using retrieval and answer-quality metrics.

## Step 12 — Inspect the first few automatically generated chunks

In [ ]:
for chunk in chunks[:5]:
    print("\n" + "=" * 72)
    print(
        f"CHUNK {chunk['chunk_id']} | "
        f"PDF page(s): {chunk['pages']} | "
        f"Length: {len(chunk['text'])}"
    )
    print("=" * 72)
    print(chunk["text"][:1200])

# Part C — Dense embeddings and FAISS retrieval

## Step 13 — Load the embedding model

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")

embedding_model = SentenceTransformer(embedding_model_name)

print("✅ Embedding model loaded")
print("Model:", embedding_model_name)

### MSc-level rationale

A dense embedding model maps textual units into a continuous vector space. Retrieval is then based on semantic proximity rather than exact lexical overlap.

The notebook normalises embeddings before indexing. For normalised vectors, inner product is equivalent to cosine similarity:

\[
\cos(x,y)=\frac{x \cdot y}{\|x\|\|y\|}
\]

This allows FAISS `IndexFlatIP` to be used as an exact cosine-style search index.

## Step 14 — Generate embeddings for all adaptive chunks

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

if not chunk_texts:
    raise ValueError(
        "❌ No chunks were generated. "
        "The PDF may require OCR."
    )

chunk_vectors = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print("✅ Chunk embeddings created")
print("Embedding matrix shape:", chunk_vectors.shape)
print("First 10 values:", chunk_vectors[0][:10])

## Step 15 — Build the FAISS vector index

In [ ]:
vector_dimension = chunk_vectors.shape[1]

faiss_index = faiss.IndexFlatIP(vector_dimension)
faiss_index.add(chunk_vectors)

print("✅ FAISS index ready")
print("Vector dimension:", vector_dimension)
print("Vectors stored  :", faiss_index.ntotal)

## Step 16 — Automatically select retrieval depth

The number of FAISS candidates and the number of chunks retained after reranking are also selected from the size of the resulting chunk collection.

In [ ]:
def choose_retrieval_depth(number_of_chunks):
    if number_of_chunks <= 10:
        candidate_k = min(5, number_of_chunks)
        final_k = min(3, candidate_k)

    elif number_of_chunks <= 50:
        candidate_k = 8
        final_k = 3

    elif number_of_chunks <= 200:
        candidate_k = 10
        final_k = 4

    else:
        candidate_k = 12
        final_k = 5

    return candidate_k, final_k


FAISS_TOP_K, FINAL_TOP_K = choose_retrieval_depth(len(chunks))

print("Automatic retrieval configuration")
print("FAISS candidates :", FAISS_TOP_K)
print("Final reranked   :", FINAL_TOP_K)

### MSc-level rationale

The first retrieval stage should generally favour **recall**, because relevant evidence discarded at this point cannot be recovered later.

The second-stage reranker then improves **precision** by selecting the strongest evidence from a broader candidate pool.

For a production system these values should be tuned against a labelled evaluation set rather than chosen solely from corpus size.

# Part D — CrossEncoder reranking

## Step 17 — Load the reranker

In [ ]:
reranker_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Loading CrossEncoder reranker...")

reranker = CrossEncoder(reranker_name)

print("✅ Reranker loaded")
print("Model:", reranker_name)

### MSc-level rationale

Dense retrieval uses a bi-encoder architecture: the query and document chunk are encoded independently. This makes large-scale retrieval efficient, but fine-grained interactions between query and passage tokens are limited.

A CrossEncoder instead receives the pair jointly:

\[
(q,d_i) \rightarrow r_i
\]

where \(r_i\) is a relevance score for candidate \(d_i\).

The two-stage architecture therefore separates:
- **efficient candidate generation** from
- **more computationally expensive relevance refinement**.

# Part E — Build the reusable RAG question-answering function

The function below shows the complete flow for every user question:

1. embed the question,
2. retrieve candidate chunks with FAISS,
3. print the first-stage ranking,
4. rerank the candidates,
5. print the second-stage ranking,
6. keep the strongest evidence,
7. construct the prompt,
8. request the final model answer,
9. print the answer and its source pages.

In [ ]:
system_prompt = (
    "You are a document-grounded question-answering assistant. "
    "Answer only from the supplied PDF context. "
    "Do not invent unsupported information. "
    "Cite evidence using labels such as [Source 1]. "
    "If the supplied evidence is insufficient, explicitly say "
    "that the answer could not be found in the uploaded PDF."
)

In [ ]:
def ask_pdf(question, show_details=True):
    question = question.strip()

    if not question:
        raise ValueError("Please enter a non-empty question.")

    # ---------------------------------------------------------
    # 1. QUESTION EMBEDDING
    # ---------------------------------------------------------
    question_vector = embedding_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # ---------------------------------------------------------
    # 2. FAISS CANDIDATE RETRIEVAL
    # ---------------------------------------------------------
    candidate_count = min(
        FAISS_TOP_K,
        faiss_index.ntotal
    )

    scores, ids = faiss_index.search(
        question_vector,
        candidate_count
    )

    candidates = []

    for score, chunk_id in zip(scores[0], ids[0]):
        candidate = chunks[int(chunk_id)].copy()
        candidate["faiss_score"] = float(score)
        candidates.append(candidate)

    if show_details:
        print("\n" + "=" * 80)
        print("STAGE 1 — FAISS CANDIDATES")
        print("=" * 80)

        for rank, item in enumerate(candidates, start=1):
            print(
                f"{rank}. Chunk {item['chunk_id']} | "
                f"Pages {item['pages']} | "
                f"FAISS score {item['faiss_score']:.4f}"
            )

    # ---------------------------------------------------------
    # 3. CROSSENCODER RERANKING
    # ---------------------------------------------------------
    pairs = [
        [question, candidate["text"]]
        for candidate in candidates
    ]

    reranker_scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, reranker_scores):
        candidate["reranker_score"] = float(score)

    candidates.sort(
        key=lambda item: item["reranker_score"],
        reverse=True
    )

    if show_details:
        print("\n" + "=" * 80)
        print("STAGE 2 — AFTER RERANKING")
        print("=" * 80)

        for rank, item in enumerate(candidates, start=1):
            print(
                f"{rank}. Chunk {item['chunk_id']} | "
                f"Pages {item['pages']} | "
                f"FAISS {item['faiss_score']:.4f} | "
                f"Reranker {item['reranker_score']:.4f}"
            )

    selected = candidates[:min(FINAL_TOP_K, len(candidates))]

    # ---------------------------------------------------------
    # 4. CONTEXT AUGMENTATION
    # ---------------------------------------------------------
    context_blocks = []

    for source_number, item in enumerate(selected, start=1):
        context_blocks.append(
            f"[Source {source_number} | "
            f"PDF page(s) {item['pages']} | "
            f"Chunk {item['chunk_id']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    if show_details:
        print("\n" + "=" * 80)
        print("STAGE 3 — CONTEXT SENT TO THE MODEL")
        print("=" * 80)
        print(context)

    # ---------------------------------------------------------
    # 5. GENERATION
    # ---------------------------------------------------------
    user_prompt = f'''
PDF CONTEXT

{context}

QUESTION

{question}

Answer the question using only the supplied PDF context.
'''.strip()

    response = llm_client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print("\n" + "=" * 80)
    print("FINAL ANSWER")
    print("=" * 80)
    print(answer)

    print("\nSources used:")

    for source_number, item in enumerate(selected, start=1):
        print(
            f"Source {source_number}: "
            f"PDF page(s) {item['pages']} | "
            f"Chunk {item['chunk_id']} | "
            f"Reranker {item['reranker_score']:.4f}"
        )

    return {
        "question": question,
        "answer": answer,
        "sources": selected
    }

### MSc-level interpretation

The function deliberately exposes intermediate retrieval decisions rather than returning only the final answer. This supports error analysis.

For example, an incorrect final answer may originate from:
- poor PDF extraction,
- unsuitable chunk boundaries,
- weak embedding retrieval,
- reranker failure,
- insufficient evidence,
- or generation that is not faithful to the supplied evidence.

A transparent pipeline helps distinguish retrieval failure from generation failure.

# Part F — Ask your own question

This is the section you can use repeatedly after the PDF has been indexed.

Run the cell below. Colab will display an input box in the cell output.

Type **any question about your uploaded PDF** and press Enter.

The notebook will then:
- retrieve evidence,
- rerank it,
- show the selected context,
- call the model,
- and print the final answer.

In [ ]:
my_question = input(
    "Enter your question about the uploaded PDF: "
).strip()

result = ask_pdf(
    question=my_question,
    show_details=True
)

## Ask another question without rebuilding the index

You do **not** need to re-run PDF extraction, chunking, embeddings, or FAISS for every new question.

Simply run this cell again.

In [ ]:
another_question = input(
    "Ask another question: "
).strip()

another_result = ask_pdf(
    question=another_question,
    show_details=True
)

## Optional — Ask a question with less diagnostic output

For normal usage you may prefer to hide the intermediate rankings and display mainly the final answer.

In [ ]:
quick_question = input(
    "Enter a question for concise mode: "
).strip()

quick_result = ask_pdf(
    question=quick_question,
    show_details=False
)

# Part G — What has become adaptive?

This version no longer relies on a single pre-defined chunk size.

The notebook automatically examines the uploaded PDF and selects parameters from:

- document page count,
- total character count,
- estimated token count,
- page density,
- paragraph count,
- median paragraph size,
- final number of generated chunks.

It then adapts:

```text
target chunk size
overlap
FAISS candidate count
final reranked context count
```

This makes the system more responsive to the characteristics of the document while preserving a transparent rule-based design.

## MSc-level limitations and evaluation considerations

Adaptive chunking should not be assumed to be optimal merely because it is automatic.

The current strategy is a deterministic engineering heuristic. To establish whether it actually improves retrieval quality, it should be evaluated against alternatives using a representative question-answer dataset.

Useful metrics include:

- **Recall@K** — whether relevant evidence appears in the retrieved candidate set.
- **MRR / nDCG** — quality of evidence ranking.
- **Context precision** — proportion of retrieved context that is actually relevant.
- **Context recall** — proportion of required evidence successfully retrieved.
- **Faithfulness** — whether the generated answer is supported by the selected evidence.
- **Answer relevance** — whether the generated answer directly addresses the question.
- **Citation correctness** — whether the cited passage supports the generated claim.

A strong Master's-level experiment would compare:

1. fixed-size chunking,
2. the adaptive heuristic implemented here,
3. semantic chunking,

while holding the embedding model, reranker, question set, and generator constant.

# Final system summary

```text
Uploaded PDF
    ↓
Automatic document profile
    ↓
Adaptive sentence-aware chunks
    ↓
MiniLM embeddings
    ↓
FAISS exact vector search
    ↓
Automatic candidate depth
    ↓
CrossEncoder reranking
    ↓
Best evidence chunks
    ↓
OpenAI-compatible client
    ↓
Grounded answer
```

### Models/components used

**Embedding model**

```text
sentence-transformers/all-MiniLM-L6-v2
```

**Reranker**

```text
cross-encoder/ms-marco-MiniLM-L-6-v2
```

**Generation client**

```python
from openai import OpenAI
```

The present credential is a DeepSeek API key, so the configured endpoint must use a DeepSeek-supported model. A genuine OpenAI GPT model requires a separate OpenAI API key.